# TAMER OCR — Colab/Kaggle Training Notebook

This notebook is an **execution layer only**. All training logic lives in the `tamer_ocr/` codebase.

What this notebook does:
1. **Pull** the codebase from GitHub
2. **Install** dependencies
3. **Run** training via `Trainer.run()`
4. **Save** outputs to Hugging Face
5. **Manage** checkpoints on Google Drive

All training logic (data processing, model architecture, training loop, checkpointing)
resides in the Python codebase — NOT in this notebook.

## 1. Environment Setup

In [ ]:
# Mount Google Drive for persistent checkpoint storage
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Check GPU
!nvidia-smi

## 2. Pull Codebase

In [ ]:
import os

# Option A: Clone from GitHub (replace with your repo URL)
GITHUB_REPO = "https://github.com/YOUR_USERNAME/tamer-ocr.git"

# Option B: Upload tamer_ocr.zip directly to Colab and unzip
# Uncomment the method you prefer:

# --- Method A: Git clone ---
# if os.path.exists('tamer-ocr'):
#     !rm -rf tamer-ocr
# !git clone $GITHUB_REPO
# %cd tamer-ocr

# --- Method B: Upload ZIP ---
# Upload your tamer_ocr.zip to Colab, then:
# !unzip -o tamer_ocr.zip -d .

# --- Method C: Already in working directory ---
# If you've already uploaded the codebase, just verify:
assert os.path.exists('tamer_ocr'), "tamer_ocr/ directory not found!"
assert os.path.exists('train.py'), "train.py not found!"
print("Codebase verified!")

## 3. Install Dependencies

In [ ]:
!pip install -q timm albumentations datasets huggingface_hub requests tqdm

## 4. Configure Training

In [ ]:
from tamer_ocr.config import Config

config = Config()

# === PATHS ===
config.data_dir = "./data"
config.output_dir = "./outputs"
config.checkpoint_dir = "/content/checkpoints"
config.log_dir = "./logs"
config.drive_backup_dir = "/content/drive/MyDrive/tamer_checkpoints"

# === MODEL ===
# (defaults are correct: swin_base, d_model=512, 6-layer decoder)

# === TRAINING ===
config.batch_size = 8
config.accumulation_steps = 4  # Effective batch = 32
config.num_epochs = 150
config.encoder_lr = 1e-5
config.decoder_lr = 1e-4
config.label_smoothing = 0.1
config.max_grad_norm = 1.0

# === SCHEDULER ===
config.pct_start = 0.1  # 10% warmup
config.total_training_steps = 50000  # ~72 hours on T4

# === TEMPERATURE SAMPLING ===
config.temp_start = 0.8  # Upweight small datasets early
config.temp_end = 0.4    # More uniform later

# === INFERENCE ===
config.beam_width = 5
config.length_penalty = 0.6

# === CHECKPOINTING ===
config.checkpoint_every_steps = 1000
config.keep_last_n_checkpoints = 3

# === HUGGING FACE ===
# Set these via Colab secrets or environment variables
import os
config.hf_repo_id = os.getenv("HF_REPO_ID", "")
config.hf_token = os.getenv("HF_TOKEN", "")

# === KAGGLE (for dataset downloads) ===
# Set via Colab secrets or environment variables
# os.environ['KAGGLE_USERNAME'] = os.getenv("KAGGLE_USERNAME", "")
# os.environ['KAGGLE_KEY'] = os.getenv("KAGGLE_KEY", "")

print("Configuration ready!")
print(f"  Device: {'CUDA' if __import__('torch').cuda.is_available() else 'CPU'}")
print(f"  Effective batch size: {config.batch_size * config.accumulation_steps}")
print(f"  Total training steps: {config.total_training_steps}")

## 5. Run Training

All training logic is in `tamer_ocr.core.trainer.Trainer`.
This cell just calls `trainer.run()` — that's it.

In [ ]:
from tamer_ocr.core.trainer import Trainer

# Create trainer — all logic is inside
trainer = Trainer(config)

# Run the full pipeline: data prep -> model build -> train -> eval
trainer.run()

## 6. Resume Training (if session crashed)

If your Colab session disconnected, run this cell to resume from the latest checkpoint.

In [ ]:
import glob
import os

# Find latest checkpoint on Drive
drive_dir = "/content/drive/MyDrive/tamer_checkpoints"
local_dir = "/content/checkpoints"

def find_latest_checkpoint(search_dir):
    if not os.path.exists(search_dir):
        return None
    ckpts = sorted(
        glob.glob(os.path.join(search_dir, "*.pt")),
        key=os.path.getmtime,
        reverse=True,
    )
    return ckpts[0] if ckpts else None

# Try Drive first, then local
latest = find_latest_checkpoint(drive_dir) or find_latest_checkpoint(local_dir)

if latest:
    print(f"Resuming from: {latest}")
    trainer = Trainer(config)
    trainer.run(resume_from=latest)
else:
    print("No checkpoint found. Starting from scratch.")
    trainer = Trainer(config)
    trainer.run()

## 7. Beam Search Evaluation

Run full beam search evaluation on the best checkpoint.

In [ ]:
# Evaluate with beam search on the best model
best_path = os.path.join(config.checkpoint_dir, "best.pt")
if os.path.exists(best_path):
    trainer = Trainer(config)
    trainer.prepare_data()
    trainer.build_model()
    trainer.resume_from_checkpoint(best_path)
    metrics = trainer.evaluate_with_beam_search(max_samples=500)
    print("\nBeam Search Results:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")
else:
    print("No best.pt checkpoint found. Train first!")

## 8. Push Final Model to Hugging Face

In [ ]:
from tamer_ocr.utils.checkpoint import push_to_huggingface

best_path = os.path.join(config.checkpoint_dir, "best.pt")
if os.path.exists(best_path) and config.hf_repo_id:
    push_to_huggingface(best_path, config, step=0, is_best=True)
    print(f"Pushed best.pt to HuggingFace: {config.hf_repo_id}")
else:
    print("Either no best.pt found or HF repo not configured.")